# Attempt Day 1

based on session notes, the defined classes and excution happy paths. are the core outer requirements to build

In [1]:
class ParkingSpot:
    def __init__(self):
        self.occupant = None

    # contracts: true if operation worked as expected, else false.
    def assign(self, vehicle):
        if self.occupant is not None: 
            return False
        else: #is None, hence self.occupant can be set to vehicle.
            self.occupant = vehicle
            return True
    
    # check unpark for later.
    def release(self):
        if self.occupant is None: 
            return False
        else: 
            self.occupant = None
            return True
    def get_occupant(self):
        return self.occupant

# TODO: design vehicle
class Vehicle:
    def __init__(self):
        pass

# TODO: design Parking Ticket.

class ParkingTicket:
    def __init__(self):
        pass

class ParkingFloor:
    def __init__(self,size):
        self.parking_grid = [ParkingSpot() for _ in range(self.size)]
        self.size = size
        self.filled = 0

    def full(self):
        if self.size == self.filled:
            return True
        else:
            return False

    # TODO: Move controller pattern from ParkingFloor to ParkingLot
    # TODO:
    def park(self, vehicle):
        if not self.full():
            #naive parkingspot implementation:
            for parking_spot in self.parking_grid:
                if parking_spot.assign(vehicle):
                    break
            self.filled += 1
        
    def unpark(self, vehicle):
        for parking_spot in self.parking_grid:
            if parking_spot.get_occupant() == vehicle and parking_spot.release(vehicle):
                self.filled -= 1
    
class ParkingLot:
    def __init__(self):
        pass
    def park(self, vehicle):
        pass
    def unpark(self,vehicle):
        pass


### Critique After Attempt 1 (Hard, no spoilers)

Reference: `../2026-05-27-session.md` sections **2A, 2E, 4A, 5A**.

- `High:` You coded classes before locking invariants/state machine. This repeats the workflow skip called out in notes (`2B`, `5A`): you are still describing behavior, not proving correctness constraints.
- `High:` Ownership is split wrong: `ParkingFloor.park/unpark` still mutates core lifecycle while your own notes converged that lot-level transitions must be owned by `ParkingLot` (`4C`: mutation closure).
- `High:` `Ticket` is structurally empty for exit semantics. Notes explicitly challenged this (`2E`, `2G`): unpark keyed by session handle is impossible to guarantee with current model.
- `Medium:` `ParkingSpot` has occupancy but no compatibility state, despite notes saying compatibility is a first-class requirement (`1B`, `2A`, `2E`).
- `Medium:` There is no invariant guard for double-booking across the system boundary; local `assign` checks are not enough as a design claim (`5A`, `5C`).

Verdict: this attempt is implementation-first and still under-modeled at the LLD level.


## Attempt 1 Continued but refactoring

1. It seems that ParkingLot is an aggregate root since it owns one to many ParkingFloors and ParkingFloors own ParkingSpots.
2. ParkingSpot should track both compatibility and occupancy, not only which vehicle is currently there.
3. Ticket likely needs more than entry time. At minimum, think about whether the system can unpark correctly if the ticket does not identify the parked vehicle or spot.
4. ParkingLot and ParkingFloor should just be collections instead of state tracking from previous
5. Move parking controller logic to ParkingLot as the core aggregate

Run with naive search and park unpark implemenation for now.

In [ ]:
class ParkingSpot:
    def __init__(self):
        self.occupant = None

    # contracts: true if operation worked as expected, else false.
    def assign(self, vehicle):
        if self.occupant is not None: 
            return False
        else: #is None, hence self.occupant can be set to vehicle.
            self.occupant = vehicle
            return True
    
    # check unpark for later.
    def release(self):
        if self.occupant is None: 
            return False
        else: 
            self.occupant = None
            return True
    def get_occupant(self):
        return self.occupant

# TODO: design vehicle
class Vehicle:
    def __init__(self, id = None, type = None, ticket = None):
        self.id = id
        self.type = type
        self.ticket = ticket
        

# TODO: design Parking Ticket.

class ParkingTicket:
    def __init__(self, id = None, entry_time = None):
        self.id = id
        self.entry_time = entry_time

class ParkingFloor:
    def __init__(self,size):
        self.parking_grid = [ParkingSpot() for _ in range(self.size)]
        self.size = size
        self.filled = 0

    def full(self):
        if self.size == self.filled:
            return True
        else:
            return False

    # TODO: Move controller pattern from ParkingFloor to ParkingLot
    # TODO:
    def park(self, vehicle):
        if not self.full():
            #naive parkingspot implementation:
            for parking_spot in self.parking_grid:
                if parking_spot.assign(vehicle):
                    break
            self.filled += 1
        
    def unpark(self, vehicle):
        for parking_spot in self.parking_grid:
            if parking_spot.get_occupant() == vehicle and parking_spot.release(vehicle):
                self.filled -= 1
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size)]
    def park(self, vehicle):
        parked = False
        for parking_floor in self.parking_floors:
            if parking_floor.park(vehicle):
                parked = True
                break
        return parked

    def unpark(self,vehicle):
        unparked = False
        for parking_floor in self.parking_floors:
            if parking_floor.unpark(vehicle):
                unparked = True
                break
        return unparked


This is a great question because it gets to the heart of **API design, DDD aggregates, identity, and state transitions**.

Let's analyze it from first principles.

---

# 1. The core operation is a state transition

The parking lot state can be modeled as:

[
ParkingLotState = { Spots, Vehicles, Tickets }
]

The two transitions are:

```
park(vehicle)   : State → State
unpark(ticket)  : State → State
```

The question is:

> What information is the minimal, stable, authoritative identity needed to perform each transition?

---

# 2. Why park takes a Vehicle

## Intent

The user's intent is:

> "I have a vehicle. Please find a place for it."

Before parking, the vehicle has no relationship with the parking lot.

The system needs information from the vehicle:

```python
Vehicle {
    id
    type  # car, motorcycle, truck
}
```

The algorithm is:

```
park(vehicle):
    spot = find_available_spot(vehicle.type)
    occupy(spot, vehicle)
    ticket = issue_ticket(vehicle, spot, entry_time)
    return ticket
```

The **vehicle determines the constraints**.

The parking lot is solving a search problem:

```
Vehicle type
      |
      v
Find compatible spot
      |
      v
Create parking relation
```

So:

```
park(vehicle)
```

matches the direction of causality.

---

# 3. Why unpark takes a Ticket

Now the situation is reversed.

After parking, the authoritative relationship is:

```
Ticket
  |
  +-- vehicle_id
  |
  +-- spot_id
  |
  +-- entry_time
```

The ticket is a **proof of an existing parking transaction**.

Unparking asks:

> "Which parking relationship should I terminate?"

You already know the answer from the ticket.

```
unpark(ticket):
    spot = ticket.spot_id
    release(spot)
    calculate_fee(ticket)
    close(ticket)
```

This is O(1).

---

# 4. Why not unpark(vehicle)?

Imagine:

```python
unpark(vehicle)
```

The system has to ask:

```
Where is this vehicle parked?
```

So you need a lookup:

```
vehicle_id → active_ticket → spot
```

which means either:

* maintain another index
* scan all tickets
* handle duplicate plates, errors, etc.

The vehicle is not the primary identity of the parking event.

---

# 5. The deeper DDD principle: identify the aggregate by the lifecycle event

A useful rule:

> **Commands should enter an aggregate using the identity that uniquely represents the current lifecycle stage.**

Lifecycle:

```
Before parking:

Vehicle
   |
   | park()
   v

ParkingSession (Ticket)
   |
   | unpark()
   v

Completed
```

The "root identity" changes as the state machine evolves.

---

# 6. Another way to see it: create vs destroy

Parking creates a relationship:

```
Vehicle + EmptySpot
          |
          v
    ParkingSession
```

Therefore the input is the thing being introduced:

```
park(vehicle)
```

---

Unparking destroys the relationship:

```
ParkingSession
      |
      v
Vehicle + EmptySpot
```

Therefore the input is the relationship identifier:

```
unpark(ticket)
```

---

# 7. General design pattern

This appears everywhere.

## Hotel

Check-in:

```python
check_in(guest)
```

creates:

```
Reservation / Stay
```

Check-out:

```python
check_out(stay_id)
```

---

## Shopping

Create order:

```python
place_order(cart)
```

Later:

```python
cancel_order(order_id)
```

---

## Banking

Open account:

```python
open_account(customer)
```

Later:

```python
close_account(account_id)
```

---

# Compression table

| Operation type                | Input identity                   | Principle                                               |
| ----------------------------- | -------------------------------- | ------------------------------------------------------- |
| Create relationship           | External entity                  | "What is entering the system?"                          |
| Modify/terminate relationship | Relationship ID / transaction ID | "Which existing state transition is being manipulated?" |
| Query state                   | Most selective identifier        | "What lets us locate state efficiently?"                |

---

So in DDD terms, **`park(vehicle)` is a command that creates a new aggregate instance (a Parking Session/Ticket), while `unpark(ticket)` is a command against an existing aggregate instance.**

The deepest principle is:

> **Commands should be expressed in terms of the entity that owns the information necessary to make the state transition valid.**

For parking, that is the **vehicle's constraints**. For unparking, that is the **ticket's identity and history**.


What you are asking for is essentially the **design derivation process**: *how do I discover the right command boundary and identity?* This is one of the central skills in DDD and low-level design.

The chain of thought is not "what objects exist?" but rather **what state transition am I trying to control?**

---

## Step 1: Ask: What is the state machine?

Do not start with classes.

Start with:

```
Empty Spot + Vehicle
        |
        | park()
        v
Occupied Spot + Parking Session
        |
        | unpark()
        v
Empty Spot + Vehicle exits
```

Now you see there is a new thing created:

```
Parking Session (Ticket)
```

This is the thing whose lifecycle matters.

---

## Step 2: Ask: What information is missing before the transition?

For `park`:

Current state:

```
ParkingLot
  spots: empty
```

Input:

```
Vehicle(type=Car)
```

Question:

> What do I need to decide whether this transition is valid?

You need:

* vehicle type
* vehicle identity (for tracking)

You **do not have a ticket yet**.

So the command must be:

```python
park(vehicle)
```

---

## Step 3: Ask: What identity is created by the transition?

After parking:

```
Ticket {
    ticket_id
    vehicle_id
    spot_id
    entry_time
}
```

The parking operation creates a **new relationship**:

```
Vehicle <---- Ticket ----> Spot
```

The ticket becomes the canonical identity of this parking session.

---

## Step 4: Ask: What does the reverse transition need to reference?

For `unpark`:

The intent is:

> "End a specific parking session."

Now ask:

**What uniquely identifies a parking session?**

Candidates:

### Vehicle

```
vehicle_id → active ticket → spot
```

Requires an additional lookup.

Also has ambiguity:

```
Vehicle A parked Monday
Vehicle A parked Tuesday
```

Which parking event?

---

### Spot

```
unpark(spot_id)
```

Question:

> Who is authorized to release this spot?

Anyone who knows the spot number can do it.

Bad domain boundary.

---

### Ticket

```
ticket_id → spot, vehicle, entry time
```

The ticket directly points to the exact relationship.

Therefore:

```python
unpark(ticket)
```

is the natural command.

---

## Step 5: Ask: What is the aggregate's current identity?

This is a very DDD way to think.

The "identity of the thing under control" changes over time.

Before parking:

```
Vehicle
```

After parking:

```
ParkingSession
```

So:

```
park(vehicle)
       |
       v
ParkingSession(ticket)
       |
unpark(ticket)
```

---

## The general design algorithm

Whenever designing an API, run this checklist:

### 1. What state transition is occurring?

```
Before ---- command ----> After
```

---

### 2. What entity is being created or destroyed?

```
A + B → Relationship
```

or

```
Relationship → A + B
```

---

### 3. What information is necessary to validate the transition?

Examples:

* vehicle type?
* account balance?
* reservation date?

---

### 4. After the transition, what becomes the authoritative identity?

Examples:

```
Order ID
Ticket ID
Reservation ID
Session ID
```

---

### 5. For future commands, use the identity of the current lifecycle stage.

---

## Compression rule

A very powerful heuristic:

| Operation                        | Usually takes                         |
| -------------------------------- | ------------------------------------- |
| Create a relationship            | The participating entities            |
| Change an existing relationship  | The relationship's ID                 |
| Destroy an existing relationship | The relationship's ID                 |
| Query                            | The smallest authoritative identifier |

Examples:

```
placeOrder(cart)         → Order
cancelOrder(orderId)

bookFlight(customer)     → Reservation
cancelFlight(reservationId)

login(credentials)       → Session
logout(sessionId)

park(vehicle)            → Ticket
unpark(ticket)
```

---

The biggest mindset shift is:

**Don't design around nouns ("I have Vehicle and Spot, so methods should take Vehicle"). Design around transitions in a state machine. The parameter is the information needed to move from one valid state to another, and over time the authoritative identity often changes.**

This is the same mental model used in DDD aggregates, event sourcing, finite state machines, and even control theory (commands acting on a system state).


## Attempt 2

### requirements:

1. Park and unpark car tracking only, no fees for now

In [ ]:

from enum import Enum
import uuid

class VehicleType(Enum):
    Bike = "Bike"
    Car = "Car"
    Truck = "Truck"

class ParkingSpot:
    def __init__(self):
        self.occupant = None
        self.type = None

    # contracts: true if operation worked as expected, else false.
    def assign(self, vehicle):
        if self.occupant is None and vehicle.type == self.type:
            self.occupant = vehicle
            return True
        else:
            return False
    
    # check unpark for later.
    def release(self, vehicle):
        if self.occupant == vehicle:
            self.occupant = None
            return True
        else:
            return False
    
    def get_occupant(self):
        return self.occupant

# TODO: design vehicle
class Vehicle:
    def __init__(self, id = None, type = None, ticket = None):
        self.id = id
        self.type = type
        self.ticket = ticket
        

# TODO: design Parking Ticket.

class ParkingTicket:
    def __init__(self, spot_id = None, vehicle_id = None, entry_time = None, floor_id = None):
        self.id = str(uuid.uuid4())
        self.spot_id = spot_id
        self.vehicle_id = vehicle_id
        self.entry_time = entry_time
        self.floor_id = floor_id


class ParkingFloor:
    def __init__(self,size):
        self.parking_grid = {spot_id: ParkingSpot(type = 'truck' if spot_id % 2 == 0 else 'car') for spot_id in range(size//2)}
        self.size = size
        self.filled = 0

    def full(self):
        return self.filled == self.size

    def park(self, vehicle):
        if not self.full():
            parking_spot = None
            for spot_id, spot in self.parking_grid.items():
                if spot.assign(vehicle):
                    parking_spot = spot_id
                    break
            if parking_spot is not None:
                self.filled += 1
                ticket = ParkingTicket(spot_id = parking_spot, vehicle_id = vehicle.id, entry_time = int(time.time()), id=uuid.uuid4())
                self.parking_grid[parking_spot].occupant.ticket = ticket
                return ticket

    def unpark(self, ticket):
        parking_spot = self.parking_grid.get(ticket.spot_id)
        if parking_spot and parking_spot.release(parking_spot.get_occupant()):
            self.filled -= 1
            del parking_spot.occupant.ticket
            return True
        return False
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size)]
        self.tickets = {}
    def park(self, vehicle):
        parked = False
        for parking_floor in self.parking_floors:
            ticket = parking_floor.park(vehicle)
            if ticket:
                parked = True
                self.tickets[ticket.id] = ticket
                break
        return parked

    def unpark(self,ticket):
        if ticket.id in self.tickets:
            unparked = False
            for parking_floor in self.parking_floors:
                if parking_floor.unpark(self.tickets[ticket.id]):
                    unparked = True
                    del self.tickets[ticket.id]
                    break
            return unparked
        return False


### Critique After Attempt 2 (latest)

1. Findings

- High: Requirements are narrowed to "park/unpark car tracking only, no fees," but the implementation still declares Bike/Car/Truck without a clear scope contract. Lock scope explicitly (single type vs multi-type) before modeling entities.
- High: Invariants are missing as always-true statements. Right now correctness is implicit in methods, so there is no explicit guarantee for core rules like "one active ticket per parked vehicle" and "one occupied vehicle per spot." 
- High: Ownership is still unstable. `ParkingFloor` performs lifecycle mutation (`park/unpark`) while `ParkingLot` also owns global active-ticket state. This split makes invariant enforcement ambiguous.
- High: The state machine is not explicit. Legal/illegal transitions (e.g., unpark with stale/forged ticket, double unpark, park when full) are not modeled first, so behavior is ad hoc in method branches.
- High: Core model has execution-breaking mismatches: `ParkingSpot.__init__` takes no `type` but is called with `type=...`; `time` is used but not imported; `ParkingTicket` constructor is called with unsupported `id` argument; `ParkingFloor(size)` creates only `size//2` spots but `full()` compares against `size`.
- Medium: Ticket identity flow is inconsistent. `ParkingLot.unpark` accepts a ticket object, but true authority should be `ticket_id -> active session` lookup; current API makes forged object usage easier and weakens boundary checks.
- Medium: `Vehicle.ticket` as mutable attached state is not clearly owned. During unpark, ticket cleanup mutates occupant internals directly (`del ...ticket`), which couples spot release with vehicle mutation authority.
- Medium: DS/operations are only partially aligned. Linear scan is fine for now, but "nearest available spot" is a requirement and is not represented in structure or ordering policy.
- Medium: Happy/failure traces are missing for this latest attempt revision, so ownership and transition gaps are not validated end-to-end.

2. Revision order

1. Rewrite section 1 with explicit scope contract for this iteration (single floor? multi-floor? which vehicle types? no fee yet).
2. Add 3 to 5 invariants as always-true statements, each with an owner that enforces it.
3. Write the spot/session state machine with legal and illegal transitions before class edits.
4. Rewrite responsibilities in `Rule -> Owner -> Mutator -> Enforcement` form, especially ticket lifecycle and unpark authority.
5. Refactor API boundary to `unpark(ticket_id)` and centralize active-session validation in `ParkingLot`.
6. Re-run one happy path and one failure path trace, then patch code to match those traces.
